In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()
test = test.sample(50)

test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 102 to 248
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    50 non-null     object
 1   label   50 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.2+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
102,exceptionally well acted by diane lane and ric...,positive
902,enchanted with low-life tragedy and liberally ...,negative
691,"the beautiful , unusual music is this film's c...",negative
358,what sets this romantic comedy apart from most...,positive
595,"slow , silly and unintentionally hilarious .",negative
604,done in mostly by a weak script that can't sup...,negative
161,pete's screenplay manages to find that real na...,positive
971,blade ii has a brilliant director and charisma...,negative
850,"the trouble is , its filmmakers run out of cle...",negative
1040,it's a feel-bad ending for a depressing story ...,negative


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_19520\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


49242112

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "gemma3",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()

    if 'positive' in content:
        content = 'positive'
    elif 'negative' in content:
        content = 'negative'
    else:
        content = 'error'

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [7]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_19520\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [8]:
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
102,exceptionally well acted by diane lane and ric...,positive,positive,4.347816,5083,119.792969,2.303979
902,enchanted with low-life tragedy and liberally ...,negative,negative,2.106595,5077,123.441406,0.079607
691,"the beautiful , unusual music is this film's c...",negative,negative,2.245559,5062,127.335938,0.219199
358,what sets this romantic comedy apart from most...,positive,positive,2.300601,5061,127.609375,0.261090
595,"slow , silly and unintentionally hilarious .",negative,positive,2.314403,5057,128.089844,0.274393
604,done in mostly by a weak script that can't sup...,negative,negative,2.225858,5066,128.148438,0.179189
161,pete's screenplay manages to find that real na...,positive,positive,2.378799,5066,128.207031,0.343485
971,blade ii has a brilliant director and charisma...,negative,negative,2.289411,5066,128.281250,0.245891
850,"the trouble is , its filmmakers run out of cle...",negative,negative,2.274170,5066,128.437500,0.229144
1040,it's a feel-bad ending for a depressing story ...,negative,negative,2.361054,5066,128.101562,0.312237


In [9]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.840000
F1 score: 0.838454
Precision: 0.862000
Recall: 0.840000


In [10]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.36167245388031
Average VRAM usage: 5066.2
Average RAM usage: 117.765078125
Average total time: 0.318099008


In [11]:
# save results to txt
with open('results/gemma_ZS_binary2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')